In [ ]:
# !rm ../data/playground-series-s6e1.zip
# !rm -rf ../data/playground-series-s6e1
# !kaggle competitions download -c playground-series-s6e1 -p ../data
# !unzip ../data/playground-series-s6e1.zip -d ../data/playground-series-s6e1
# !rm ../data/playground-series-s6e1.zip

zsh:1: command not found: rm
zsh:1: command not found: rm
100%|██████████████████████████████████████| 13.8M/13.8M [00:02<00:00, 5.02MB/s]

zsh:1: command not found: unzip
zsh:1: command not found: rm


In [13]:
import pandas as pd
# build a neural network with PyTorch to predict exam_score from the other columns
import torch
import torch.nn as nn
import torch.optim as optim

# split into train and test
from sklearn.model_selection import train_test_split
import torch
# standardize study_hours, class_attendance, sleep_hours, exam_score to have mean 0 and std 1
from sklearn.preprocessing import StandardScaler

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [14]:
df = pd.read_csv("../data/playground-series-s6e1/train.csv")
df.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [15]:
# find missing values
df.isnull().sum()

id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [16]:
NUMERIC_COLS = ["study_hours", "class_attendance", "sleep_hours"]


def preprocess(df, scaler=None):
    """
    Encode and scale features.
    scaler=None  → fit a new scaler (training)
    scaler=...   → apply existing scaler (inference/submission)
    Returns (processed_df, scaler)
    """
    df = df.copy()
    if "id" in df.columns:
        df = df.drop("id", axis=1)

    df["age"] = df["age"].astype(int)
    df["is_female"] = df["gender"].apply(lambda x: 1 if x == "female" else 0)
    df = df.drop("gender", axis=1)
    df = pd.get_dummies(df, columns=["course"], prefix="course", dtype=int)
    df["has_internet"] = df["internet_access"].apply(lambda x: 1 if x == "yes" else 0)
    df = df.drop("internet_access", axis=1)
    df["sleep_quality"] = df["sleep_quality"].map({"poor": 0, "average": 1, "good": 2})
    df = pd.get_dummies(df, columns=["study_method"], prefix="study_method", dtype=int)
    df["facility_rating"] = df["facility_rating"].map(
        {"low": 0, "medium": 1, "high": 2}
    )
    df["exam_difficulty"] = df["exam_difficulty"].map(
        {"easy": 0, "moderate": 1, "hard": 2}
    )

    if scaler is None:
        scaler = StandardScaler()
        df[NUMERIC_COLS] = scaler.fit_transform(df[NUMERIC_COLS])
    else:
        df[NUMERIC_COLS] = scaler.transform(df[NUMERIC_COLS])

    return df, scaler

In [17]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df, scaler = preprocess(train_df)
test_df, _ = preprocess(test_df, scaler=scaler)

y_train = train_df["exam_score"]
X_train = train_df.drop("exam_score", axis=1)
y_test = test_df["exam_score"]
X_test = test_df.drop("exam_score", axis=1)


In [18]:

hidden1_size = 128
hidden2_size = 64
hidden3_size = 32
hidden3_size = 16
hidden2_size = 8


class StudentScorePredictor(nn.Module):
    def __init__(self, input_size):
        super(StudentScorePredictor, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1_size)
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.fc3 = nn.Linear(hidden2_size, hidden3_size)
        self.fc4 = nn.Linear(hidden3_size, hidden3_size)
        self.fc5 = nn.Linear(hidden3_size, hidden2_size)
        self.fc6 = nn.Linear(hidden2_size, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = self.fc6(x)
        return x


input_size = X_train.shape[1]
model = StudentScorePredictor(input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# convert X_train and y_train to torch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)


train_dataloader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=10000,
    shuffle=True,
)


In [19]:

# train the model for 5000 epochs
num_epochs = 5000
for epoch in range(num_epochs):
    model.train()
    for X_batch, y_batch in train_dataloader:
        optimizer.zero_grad()
        y_hat = model(X_batch)
        loss = criterion(y_hat, y_batch)
        loss.backward()
        optimizer.step()
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}", end="\r")
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

# evaluate the model on the test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)
    print(f"Test Loss: {test_loss.item():.4f}")


Epoch [10/5000], Loss: 94.14453
Epoch [20/5000], Loss: 84.2142
Epoch [30/5000], Loss: 79.7950
Epoch [40/5000], Loss: 77.2628
Epoch [50/5000], Loss: 80.0089
Epoch [60/5000], Loss: 79.0826
Epoch [70/5000], Loss: 79.8381
Epoch [80/5000], Loss: 78.2425
Epoch [90/5000], Loss: 76.2546
Epoch [100/5000], Loss: 76.6128
Epoch [110/5000], Loss: 77.1776
Epoch [120/5000], Loss: 80.8339
Epoch [130/5000], Loss: 79.9955
Epoch [140/5000], Loss: 77.5425
Epoch [150/5000], Loss: 79.7214
Epoch [160/5000], Loss: 79.4172
Epoch [170/5000], Loss: 77.7579
Epoch [180/5000], Loss: 75.1640
Epoch [190/5000], Loss: 77.7592
Epoch [200/5000], Loss: 80.4207
Epoch [210/5000], Loss: 80.3770
Epoch [220/5000], Loss: 78.0598


KeyboardInterrupt: 

In [ ]:
# build a sample submission file from the example submission file ./data/playground-series-s6e1/sample_submission.csv
sample_submission = pd.read_csv("../data/playground-series-s6e1/sample_submission.csv")
sample_submission.head()

,id,exam_score
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [ ]:
submission_df = pd.read_csv("../data/playground-series-s6e1/test.csv")
ids = submission_df["id"]

submission_features, _ = preprocess(submission_df, scaler=scaler)

model.eval()
with torch.no_grad():
    sub_tensor = torch.tensor(submission_features.values, dtype=torch.float32)
    sub_preds = model(sub_tensor).numpy().flatten()

pd.DataFrame({"id": ids, "exam_score": sub_preds}).to_csv(
    "./data/playground-series-s6e1/submission.csv", index=False
)


In [96]:
!kaggle competitions submit -c playground-series-s6e1 -f ./data/playground-series-s6e1/submission.csv -m "Message"

100%|███████████████████████████████████████| 4.22M/4.22M [00:06<00:00, 720kB/s]
Successfully submitted to Predicting Student Test Scores